# Evaluation Metrics for Classification Models

**Objective:** Understand and implement common evaluation metrics for classification models using Python and Scikit-learn.

This experiment trains a classifier on the Breast Cancer Wisconsin dataset, evaluates its predictions using several metrics, and explains how to interpret the results. Run the cells in order.


## 1. Import libraries

We use Scikit-learn for the dataset, model, and metrics. Matplotlib and Seaborn are used to visualize the confusion matrix and ROC curve.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    roc_curve
)

RANDOM_STATE = 42


## 2. Load and inspect the dataset

The dataset contains measurements from breast tumor samples. The target labels are 0 = malignant and 1 = benign. For this experiment, we treat **malignant (0)** as the positive class so that recall measures how many malignant cases are detected.


In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
# Convert target to 1 for malignant (positive class), 0 for benign.
y = (data.target == 0).astype(int)

print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print("Class counts (0 = benign, 1 = malignant):")
print(y.value_counts().sort_index())
X.head()


## 3. Split into training and test sets

Stratification preserves the class proportions in each split. The test set is held out until evaluation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


## 4. Train a Logistic Regression model

Standardization is included in a pipeline so it is fitted only on the training data and applied consistently to the test data.


In [ ]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print("Model training complete.")


## 5. Confusion matrix and its components

For this binary task, positive means malignant.

- **TP:** malignant samples correctly predicted as malignant
- **TN:** benign samples correctly predicted as benign
- **FP:** benign samples incorrectly predicted as malignant
- **FN:** malignant samples incorrectly predicted as benign


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print(f"TN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}")

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Benign (0)", "Malignant (1)"],
    yticklabels=["Benign (0)", "Malignant (1)"]
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## 6. Calculate classification metrics

The metrics answer different questions. **Accuracy** is the overall fraction of correct predictions. **Precision** measures the proportion of predicted malignant cases that are truly malignant. **Recall/sensitivity** measures the proportion of malignant cases detected. **Specificity** measures the proportion of benign cases correctly recognized. **F1-score** is the harmonic mean of precision and recall. **ROC-AUC** summarizes discrimination across probability thresholds.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) else 0.0
roc_auc = roc_auc_score(y_test, y_prob)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall / Sensitivity", "Specificity", "F1-score", "ROC-AUC"],
    "Value": [accuracy, precision, recall, specificity, f1, roc_auc]
})
metrics.style.format({"Value": "{:.4f}"})


## 7. Classification report

The report shows precision, recall, F1-score, and support for each class. Since our positive label is malignant, verify that the class labels are interpreted correctly.


In [ ]:
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Benign", "Malignant"],
    zero_division=0,
    digits=4
))


## 8. ROC curve

The ROC curve plots the true positive rate against the false positive rate as the decision threshold changes. A curve closer to the top-left generally indicates better separation.


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 9. Change the probability threshold

The default classifier predicts the positive class at a probability threshold of 0.5. Lowering the threshold may increase malignant-case recall while also increasing false positives.


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_rows = []
for threshold in [0.30, 0.50, 0.70]:
    pred_at_threshold = (y_prob >= threshold).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, pred_at_threshold, labels=[0, 1]).ravel()
    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, pred_at_threshold, zero_division=0),
        "Recall": recall_score(y_test, pred_at_threshold, zero_division=0),
        "Specificity": tn_t / (tn_t + fp_t) if (tn_t + fp_t) else 0.0,
        "F1-score": f1_score(y_test, pred_at_threshold, zero_division=0),
        "False positives": fp_t,
        "False negatives": fn_t
    })
threshold_results = pd.DataFrame(threshold_rows)
threshold_results.style.format({
    "Threshold": "{:.2f}", "Precision": "{:.3f}", "Recall": "{:.3f}",
    "Specificity": "{:.3f}", "F1-score": "{:.3f}"
})


## 10. Interpretation

Write a short interpretation after running the notebook:

1. Which metric is most important when missing a malignant case is costly? Why?
2. How do false positives and false negatives change as the threshold increases?
3. Why can accuracy alone be misleading when class counts are unequal?

**Expected discussion:** In a screening context, recall for malignant cases is often important because false negatives represent missed cases. However, lowering the threshold can increase false positives. The appropriate trade-off depends on the application and should be chosen with domain experts.


## 11. Conclusion

This experiment trained a Logistic Regression classifier and evaluated it using the confusion matrix, accuracy, precision, recall, specificity, F1-score, ROC-AUC, classification report, and ROC curve. It also demonstrated how changing the decision threshold affects classification outcomes.


## Student implementation questions

1. Train Decision Tree and Random Forest classifiers on the same train/test split. Compare their accuracy, precision, recall, specificity, F1-score, and ROC-AUC with Logistic Regression. Display confusion matrices and identify which model best detects malignant cases.

2. Repeat the threshold analysis using a class-imbalanced dataset generated with `make_classification`. Compare precision, recall, specificity, and F1-score at thresholds 0.2, 0.5, and 0.8. Plot the metrics and explain the changes in false positives and false negatives.
